In [ ]:
import sys
from pathlib import Path

_SRC = Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from fraud_detect import config, viz, models, evaluation as ev, error_analysis as ea, tuning
from fraud_detect.models import ModelBackend

warnings.filterwarnings("ignore")
viz.configure_style()

## 14. Error Analysis

Understand *where* the model fails: error rates by segment,
feature distribution shifts, confident false positives/negatives,
and patterns across transaction amounts and time.

> Logic in `fraud_detect.error_analysis`

In [ ]:
path = config.PROCESSED_TRAIN_PATH if config.PROCESSED_TRAIN_PATH.exists() else config.MERGED_TRAIN_PATH
train = pd.read_parquet(path)
split = models.make_train_val_split(train)
print(f"Train: {split.X_train.shape}, Val: {split.X_val.shape}")

### 14.1 Load Best Model

In [ ]:
best_params = tuning.load_best_params(ModelBackend.LIGHTGBM, fallback_to_defaults=True)
best_result = models.train_model(train, backend=ModelBackend.LIGHTGBM, params=best_params)

y_pred_proba = best_result.model.predict_proba(split.X_val)[:, 1]
y_true = split.y_val

threshold, youden = ev.find_best_threshold(y_true, y_pred_proba)
y_pred = (y_pred_proba >= threshold).astype(int)
print(f"LightGBM | AUC: {best_result.val_auc:.5f} | Threshold: {threshold:.4f} | Youden: {youden:.4f}")

### 14.2 Overall Error Profile

In [ ]:
profile = ea.compute_error_profile(split.X_val, y_true, y_pred)

overall = profile.overall
print(f"Error rate: {overall['error_rate']:.4f} | FP: {overall['fp']} | FN: {overall['fn']}")
print(f"Worst segments:")
profile.worst_segments.head(10)

### 14.3 Error Rate by Category

In [ ]:
fig, ax = viz.plot_error_rate_by_category(profile, "ProductCD", top_n=10)
viz.save_figure(fig, config.METADATA_DIR / "error_rate_by_productcd.png")
plt.show()

In [ ]:
# Error rate by DeviceType
if "DeviceType" in split.X_val.columns:
    fig, ax = viz.plot_error_rate_by_category(profile, "DeviceType")
    viz.save_figure(fig, config.METADATA_DIR / "error_rate_by_device.png")
    plt.show()

In [ ]:
# Error rate by hour (if hour feature exists)
if "hour" in split.X_val.columns:
    fig, ax = viz.plot_error_rate_by_category(profile, "hour", top_n=24, min_samples=5)
    viz.save_figure(fig, config.METADATA_DIR / "error_rate_by_hour.png")
    plt.show()

### 14.4 Feature Distribution Shift

Largest mean differences between correct and incorrect predictions.
Suggests where the model struggles.

In [ ]:
shift = ea.feature_distribution_shift(split.X_val, y_true, y_pred)
if not shift.empty:
    shift.head(15)

In [ ]:
if not shift.empty:
    fig, ax = viz.plot_feature_shift_comparison(shift, top_n=15)
    viz.save_figure(fig, config.METADATA_DIR / "feature_shift_comparison.png")
    plt.show()

### 14.5 Confusion by Amount

In [ ]:
if "TransactionAmt" in split.X_val.columns:
    fig, ax = viz.plot_confusion_by_amount(split.X_val, y_true, y_pred)
    viz.save_figure(fig, config.METADATA_DIR / "confusion_by_amount.png")
    plt.show()
    
    amount_table = ea.confusion_by_amount_bins(split.X_val, y_true, y_pred)
    amount_table

### 14.6 Most Confident False Positives

In [ ]:
fps = ea.top_false_positives(split.X_val, y_true, y_pred_proba, n=20)
if not fps.empty:
    cols = [c for c in ["TransactionAmt", "ProductCD", "hour", "predicted_probability"] if c in fps.columns]
    fps[cols].head(10)

In [ ]:
if not fps.empty:
    fig, ax = viz.plot_false_positive_examples(fps)
    viz.save_figure(fig, config.METADATA_DIR / "false_positive_heatmap.png")
    plt.show()

### 14.7 Most Confident False Negatives

In [ ]:
fns = ea.top_false_negatives(split.X_val, y_true, y_pred_proba, n=20)
if not fns.empty:
    cols = [c for c in ["TransactionAmt", "ProductCD", "hour", "predicted_probability"] if c in fns.columns]
    fns[cols].head(10)

### 14.8 Temporal Pattern

In [ ]:
if "hour" in split.X_val.columns:
    hours = split.X_val["hour"]
    errors = (y_true != y_pred).astype(int)
    
    # Error rate by hour
    hourly_errors = pd.DataFrame({"hour": hours, "error": errors}).groupby("hour").agg(
        n_samples=("error", "count"),
        error_rate=("error", "mean"),
    ).reset_index()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(hourly_errors["hour"], hourly_errors["error_rate"], color="#c44e52", alpha=0.7)
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Error rate")
    ax.set_title("Error rate by hour of day")
    ax.set_xticks(range(24))
    plt.tight_layout()
    viz.save_figure(fig, config.METADATA_DIR / "error_rate_temporal.png")
    plt.show()

### 14.9 Summary

| Analysis | Finding |
|---|---|

Next: notebook 15 — final summary.